In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# No Databricks, a variável 'spark' já existe pronta.
# Esta linha garante que o mesmo código também rode no seu PC com `pip install pyspark`.
spark = SparkSession.builder.appName("FraudeCartaoBigData").getOrCreate()

# 1. Leitura dos dados (tabela do Unity Catalog)
df = spark.read.table("workspace.default.transacoes_simuladas")

# 2. Conversão de data/hora e criação de timestamp em segundos (necessário para rangeBetween)
df = (
    df.withColumn("TX_DATETIME", F.to_timestamp("TX_DATETIME"))
      .withColumn("TX_UNIX_TIME", F.unix_timestamp("TX_DATETIME"))
      .withColumn("TX_TIME_HOURS", F.hour("TX_DATETIME"))
      .withColumn("TX_DAY_OF_WEEK", F.dayofweek("TX_DATETIME"))
)

# 3. Definição das Janelas Temporais (Window Functions) por Cliente
# rangeBetween(-3600, 0) pega todas as transações dos últimos 3600 segundos (1h) até a atual
w_1h = (
    Window.partitionBy("CUSTOMER_ID")
          .orderBy(F.col("TX_UNIX_TIME"))
          .rangeBetween(-3600, 0)
)

w_24h = (
    Window.partitionBy("CUSTOMER_ID")
          .orderBy(F.col("TX_UNIX_TIME"))
          .rangeBetween(-86400, 0)
)

# 4. Velocity Rules (subtraímos a transação atual para olhar apenas o histórico prévio)
df_features = (
    df.withColumn("TX_COUNT_1H", F.count("TX_AMOUNT").over(w_1h) - 1)
      .withColumn("TX_SUM_1H", F.round(F.sum("TX_AMOUNT").over(w_1h) - F.col("TX_AMOUNT"), 2))
      .withColumn("TX_COUNT_24H", F.count("TX_AMOUNT").over(w_24h) - 1)
      .withColumn("TX_SUM_24H", F.round(F.sum("TX_AMOUNT").over(w_24h) - F.col("TX_AMOUNT"), 2))
)

# 5. Risk Scoring do Terminal (usando janela histórica para evitar vazamento de dados futuros)
w_terminal_hist = (
    Window.partitionBy("TERMINAL_ID")
          .orderBy(F.col("TX_UNIX_TIME"))
          .rowsBetween(Window.unboundedPreceding, -1)
)

df_features = df_features.withColumn(
    "RISCO_TERMINAL",
    F.coalesce(F.avg("FRAUD").over(w_terminal_hist), F.lit(0.0))
)

# 6. Seleção da matriz final (sem IDs arbitrários)
colunas_finais = [
    "TX_AMOUNT", "TX_TIME_HOURS", "TX_DAY_OF_WEEK",
    "TX_COUNT_1H", "TX_SUM_1H", "TX_COUNT_24H", "TX_SUM_24H",
    "RISCO_TERMINAL", "FRAUD"
]

df_final = df_features.select(colunas_finais)
df_final.show(10)

# 7. Salvamento em tabela Delta no Unity Catalog (formato colunar otimizado para Big Data)
df_final.write.mode("overwrite").saveAsTable("workspace.default.transacoes_features")

+---------+-------------+--------------+-----------+---------+------------+----------+--------------+-----+
|TX_AMOUNT|TX_TIME_HOURS|TX_DAY_OF_WEEK|TX_COUNT_1H|TX_SUM_1H|TX_COUNT_24H|TX_SUM_24H|RISCO_TERMINAL|FRAUD|
+---------+-------------+--------------+-----------+---------+------------+----------+--------------+-----+
|    30.06|            9|             6|          0|      0.0|           0|       0.0|           0.0|    0|
|    22.67|           20|             6|          0|      0.0|           0|       0.0|           0.0|    0|
|   151.25|            8|             7|          0|      0.0|           3|    360.12|           0.0|    0|
|    23.96|            9|             7|          0|      0.0|           1|     22.67|           0.0|    0|
|    150.1|           11|             7|          0|      0.0|           4|    511.37|           0.0|    0|
|    200.1|           16|             7|          1|   143.57|           2|    275.21|           0.0|    0|
|   263.18|           17|   